# Lecture 5 — Capstone: *Review, Diagnose, and Repair a Biomarker Study*
### Practical Machine Learning for Transcriptomics in Cancer Research

This is the **capstone**. You are handed a *deliberately flawed* biomarker analysis — a short "submitted
manuscript" with an impressive headline result — and you do what a careful reviewer (and a careful author)
must do:

1. **Review** the submitted analysis and reproduce its headline number.
2. **Diagnose** its methodological flaws — name each one, with evidence.
3. **Propose** corrections.
4. **Rebuild** the workflow honestly — leakage-safe, nested, externally validated.
5. **Compare** original vs corrected and quantify the inflation.
6. **Write a reviewer report** with a publish / revise / reject recommendation. *(the primary deliverable)*
7. **Reflect** on which course lessons mattered most.

> **The thesis you are demonstrating:** *the hardest part of machine learning is not training a model; it
> is demonstrating that the model is trustworthy.* Expect the impressive headline (only ~0.69 to begin with) to **unravel**
> once the flaws are fixed. **That unravelling is the lesson** — it is the whole course (validation >
> optimization) in one project.

#### Continuity & reminders
- **Label:** binary **recurrence** — *not* pCR. The binary label is a deliberate simplification of
  time-to-event data; an optional Part 4 cell restores the survival framing.
- **The flaws are not labelled in the analysis** — you discover them with the reviewer checklist.
- **Leakage discipline:** selection, scaling, *and batch correction* are fit on training data only and
  applied to held-out / external data — never pooled across the split.
- **Same data cache as Lectures 1–4** — METABRIC + GSE6532 are reused, not re-downloaded.

> **Network note.** Reuses the L1–L4 real-data loaders (cBioPortal + GEO). If the prepared cohort is in
> the shared cache it is used directly; otherwise regenerated (needs internet). Downloads are git-ignored.


## Setup

Environment and data are prepared by **Lesson 0** (conda env `ml26` + downloads). This cell only
imports libraries and the checkpoint helpers — it installs nothing.

### 📚 Official docs & resources for this lesson

New to a tool, or want the authoritative reference? These point at exactly what this notebook uses. (Lesson 0 has the full library table.)

- **Core stack** — [NumPy](https://numpy.org/doc/stable/) · [pandas](https://pandas.pydata.org/docs/) · [Matplotlib](https://matplotlib.org/stable/index.html)
- **scikit-learn** — [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline) · [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html) · [Ensembles](https://scikit-learn.org/stable/modules/ensemble.html) · [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [Metrics & scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- **Survival analysis** — [lifelines](https://lifelines.readthedocs.io/en/latest/) · [Kaplan–Meier & survival intro](https://lifelines.readthedocs.io/en/latest/Survival%20analysis%20with%20lifelines.html)
- **External cohort** — [NCBI GEO — GSE6532](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE6532) · [GEOparse](https://geoparse.readthedocs.io/en/latest/)
- **Going deeper** — [Common pitfalls & data leakage](https://scikit-learn.org/stable/common_pitfalls.html) · [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)


In [ ]:
# ── Setup — imports + checkpoint I/O (environment & data come from LESSON 0) ────
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_val_predict,
                                     train_test_split, GridSearchCV)
from sklearn.metrics import roc_auc_score, average_precision_score

# ── checkpoint I/O — the data hand-off between lessons ─────────────────────────
# Each lesson SAVES what the next one needs and LOADS what the previous one made.
# Checkpoints live in the shared datasets/derived/ folder (git-ignored).
import pickle

def _derived_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for d in sorted((base / "lessons").glob("lesson01_*/practical/task/datasets")):
            (d / "derived").mkdir(parents=True, exist_ok=True)
            return d / "derived"
    d = Path.cwd() / "datasets" / "derived"; d.mkdir(parents=True, exist_ok=True)
    return d

def save_checkpoint(name, **objs):
    path = _derived_dir() / f"{name}.pkl"
    with open(path, "wb") as fh:
        pickle.dump(objs, fh)
    print(f"saved checkpoint '{name}'  ->  {path}")
    return path

def load_checkpoint(name):
    path = _derived_dir() / f"{name}.pkl"
    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint '{name}' not found ({path}).\n"
            f"Run the earlier lesson that creates it first — it ends with "
            f"save_checkpoint('{name}', ...).")
    with open(path, "rb") as fh:
        return pickle.load(fh)


### Load features from Lesson 3, and build the external cohort

> **Reminder — this capstone builds on Lesson 3.** It **loads** the `lesson03_features` checkpoint
> (raw genes + engineered features + split), recovers the survival frame from the clinical table, and
> builds a **real external cohort** (GSE6532) here. If the load fails, run **Lesson 3** first.

In [ ]:
# The capstone needs raw genes + engineered features from LESSON 3, plus a REAL external
# cohort (GSE6532) built here. Strict hand-off: if the load raises, run Lesson 3 first.
def _resolve_data_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for cand in sorted((parent / "lessons").glob("lesson01_*/practical/task/datasets")):
            return str(cand)
    return str(Path.cwd() / "datasets")
DATA_DIR = _resolve_data_dir()

def _cached(fname):
    """Path to a file Lesson 0 downloaded; raises a clear pointer to Lesson 0 if missing."""
    p = os.path.join(DATA_DIR, fname)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        raise FileNotFoundError(f"'{fname}' not in the cache ({DATA_DIR}). Run Lesson 0 first "
                                "(lessons/lesson00_prerequisites/).")
    return p

SIGNATURES = {
    "proliferation": ["MKI67","AURKA","CCNB1","CCNB2","BUB1","TOP2A","CDK1","CCNE2","MYBL2","UBE2C","BIRC5","RRM2","TYMS","CENPF","PLK1"],
    "er_signalling": ["ESR1","FOXA1","GATA3","XBP1","BCL2","PGR","TFF1","GREB1","AR","NAT1","MLPH"],
    "immune":        ["CD8A","CD8B","GZMB","PRF1","IFNG","CXCL9","CXCL10","CD3D","CD3E","GZMA","NKG7","STAT1"],
    "stromal":       ["FAP","COL1A1","COL1A2","COL3A1","ACTA2","PDGFRB","FN1","VIM","THY1","SPARC","TIMP1"],
}
HALLMARK_SETS = {
    "E2F_TARGETS":      ["MKI67","BUB1","CCNB2","AURKA","TOP2A","RRM2","MYBL2","CDK1"],
    "G2M_CHECKPOINT":   ["CCNB1","CCNB2","PLK1","BUB1","CENPF","UBE2C","BIRC5","CDK1"],
    "ESTROGEN_EARLY":   ["ESR1","FOXA1","GATA3","TFF1","GREB1","PGR","XBP1"],
    "ESTROGEN_LATE":    ["BCL2","NAT1","MLPH","AR","ESR1","TFF1"],
    "INTERFERON_GAMMA": ["CXCL9","CXCL10","STAT1","IFNG","GZMB","PRF1"],
    "INFLAMMATORY":     ["CD8A","CD3D","CD3E","GZMA","NKG7","CD8B"],
    "EMT":              ["COL1A1","COL1A2","COL3A1","FN1","VIM","SPARC","ACTA2"],
    "ANGIOGENESIS":     ["PDGFRB","TIMP1","FAP","THY1","SPARC"],
    "APOPTOSIS":        ["BCL2","BIRC5","TIMP1","GZMB"],
    "MYC_TARGETS":      ["RRM2","TYMS","CCNE2","UBE2C","CDK1","MYBL2"],
}
def score_sets(X, sets):
    """Mean of standardised member genes -> one score per set (leakage-exempt: fixed lists)."""
    cols = {}
    for name, genes in sets.items():
        present = [g for g in genes if g in X.columns]
        if not present:
            continue
        z = (X[present] - X[present].mean()) / (X[present].std() + 1e-9)
        cols[name] = z.mean(axis=1)
    return pd.DataFrame(cols, index=X.index)

def load_gse6532_external(feature_columns, data_dir=DATA_DIR, horizon_months=60):
    """Build a REAL external-validation cohort from GSE6532 (Loi et al., Affymetrix).
    Cross-platform test (METABRIC/Illumina -> GSE6532/Affymetrix): same engineered features,
    same binary recurrence label (DMFS at the horizon), ER+ patients. Downloads ~180 MB on first run."""
    import GEOparse
    gse = GEOparse.get_GEO(filepath=_cached("GSE6532_family.soft.gz"), silent=True)
    PLATS = ("GPL96", "GPL570")
    pmap = {}
    for pl in PLATS:
        gpl = gse.gpls[pl].table
        sym = next(c for c in gpl.columns if c.lower() in ("gene symbol", "gene_symbol", "symbol"))
        m = gpl.set_index("ID")[sym].dropna().astype(str); m = m[m.str.len() > 0]
        pmap.update(m.to_dict())
    cols, meta = {}, {}
    for name, gsm in gse.gsms.items():
        if gsm.metadata.get("platform_id", ["?"])[0] not in PLATS:
            continue
        tbl = gsm.table
        if tbl is None or "VALUE" not in tbl.columns:
            continue
        cols[name] = pd.Series(tbl["VALUE"].values, index=tbl["ID_REF"].astype(str).values)
        dd = {}
        for it in gsm.metadata.get("characteristics_ch1", []):
            if ":" in it:
                k, v = it.split(":", 1); dd[k.strip().lower()] = v.strip()
        meta[name] = dd
    expr = pd.DataFrame(cols); expr = expr[expr.index.isin(pmap)]
    expr.index = [pmap[i] for i in expr.index]
    Xg = expr.groupby(level=0).mean().T
    clin = pd.DataFrame(meta).T
    er = clin["er"].eq("1")
    event = pd.to_numeric(clin.get("e.dmfs"), errors="coerce")
    months = pd.to_numeric(clin.get("t.dmfs"), errors="coerce") / 30.44
    y = pd.Series(index=clin.index, dtype="float")
    y[(event == 1) & (months <= horizon_months)] = 1
    y[(event == 0) & (months >= horizon_months)] = 0
    y[(event == 1) & (months > horizon_months)] = 0
    keep = (er & y.notna()); y = y[keep].astype(int)
    feats = pd.concat([score_sets(Xg, SIGNATURES), score_sets(Xg, HALLMARK_SETS)], axis=1).loc[y.index]
    feats = feats.reindex(columns=list(feature_columns))
    return feats, y, Xg.loc[y.index]

d = load_checkpoint("lesson03_features")
X_genes, X_feat, clin_all, y_all = d["X_genes"], d["feats"], d["clin"], d["y"]
tr, va, te = d["tr"], d["va"], d["te"]

# survival frame (time-to-event) recovered from the clinical table, for the Part-4 survival cell
_status = next(c for c in ["RFS_STATUS", "DFS_STATUS"] if c in clin_all.columns)
_months = next(c for c in ["RFS_MONTHS", "DFS_MONTHS"] if c in clin_all.columns)
surv_all = pd.DataFrame({"months": pd.to_numeric(clin_all[_months], errors="coerce"),
                         "event":  clin_all[_status].astype(str).str.startswith("1").astype(int)}).loc[y_all.index]

# REAL external cohort (downloads GSE6532 SOFT ~180 MB on first run)
X_ext_feat, y_ext, X_ext_genes = load_gse6532_external(X_feat.columns, data_dir=DATA_DIR)
print(f"raw genes: {X_genes.shape} | engineered features: {X_feat.shape[1]}")
print(f"split: train {len(tr)} | val {len(va)} | test {len(te)} | recurrence {y_all.mean():.1%}")
print(f"external GSE6532 cohort (real, Affymetrix): {len(y_ext)} ER+ patients, recurrence {y_ext.mean():.1%}")


---
## Part 1 — Review the provided analysis  *(≈20 min)*

> **Submitted manuscript (abstract).** *"We report a novel 50-gene transcriptomic signature that predicts
> recurrence in HR+/HER2− early breast cancer with cross-validated AUC ≈ 0.69. After selecting the most
> recurrence-associated genes and normalising the cohort, a gradient-boosting classifier — with
> hyperparameters optimised by cross-validation — achieves what the authors call strong, clinically promising discrimination. The top genes are
> drivers of recurrence and represent promising therapeutic targets."*

The cell below is the authors' analysis, reproduced faithfully. **Run it and reproduce their headline
number.** Do not fix anything yet — your job in Part 2 is to find out *why* this number is not what it
seems.


In [ ]:
# ================= SUBMITTED ANALYSIS (as provided by the authors) =================
# Reproduce the headline number exactly as written. (We will audit this in Part 2.)
X_all = X_genes.copy()                       # raw genes, all patients
y = y_all.copy()

# "Normalise the cohort" — scaling fit on ALL samples at once
scaler_all = StandardScaler()
X_scaled = pd.DataFrame(scaler_all.fit_transform(X_all.fillna(X_all.median())),
                        index=X_all.index, columns=X_all.columns)

# "Select the most recurrence-associated genes" — supervised selection on ALL data
selector = SelectKBest(score_func=f_classif, k=50)
selector.fit(X_scaled, y)
top50 = X_scaled.columns[selector.get_support()]
X_sig = X_scaled[top50]

# "Gradient boosting, hyperparameters optimised by CV" — tuned and reported on the SAME CV
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
grid = {"n_estimators": [100, 300], "max_depth": [2, 3], "learning_rate": [0.05, 0.1]}
search = GridSearchCV(gb, grid, scoring="roc_auc",
                      cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
search.fit(X_sig, y)

headline_auc = search.best_score_
print(f"HEADLINE: cross-validated AUC = {headline_auc:.3f}  (50-gene signature, tuned gradient boosting)")
print(f"Reported 'driver' genes (top 8 of 50): {list(top50[:8])}")
# (Authors then describe these genes as 'drivers of recurrence and therapeutic targets'.)

> **Exercise 1.1 — restate the claim.** In one or two sentences, state the analysis's claim precisely:
> the prediction task, the cohort and *n*, the metric, the reported performance, and the biological
> assertion. (You'll test each of these in Part 2.)


**Claim (your turn):** *(TODO — restate the task, cohort & n, metric, reported performance, and the biological assertion in 1–2 sentences.)*

---
## Part 2 — Identify the methodological flaws  *(≈35 min — spine)*

Audit the submitted analysis against the **reviewer checklist**. The flaws are *not* labelled; you find
them. Then demonstrate *why* the leakage inflates the number.


> **Exercise 2.1 — the flaw table.** Read the Part 1 cell line by line and identify each methodological
> flaw: what it is, *where* it occurs, why it inflates / over-claims, and which lecture it violates. Fill
> in the table in the markdown cell below (aim for five).
>
> **Exercise 2.2 — demonstrate the leakage.** Show *why* selecting genes on all the data before CV
> inflates performance: re-run the authors' select-then-CV procedure on **permuted (shuffled) labels**.
> Honest pipelines score ≈ 0.5 on permuted labels; a leaky one scores well above 0.5.


In [ ]:
# TODO 2.2 — demonstrate the leakage with a permuted-label test.
# Write a function leaky_score(y_use) that reproduces the authors' SELECT-ON-ALL-DATA-then-CV procedure
# (StandardScaler on all data -> SelectKBest(k=50) on all data -> cross_val_score of a simple model).
# Call it with the real labels, then with several PERMUTED (shuffled) label vectors.
# An honest pipeline scores ~0.50 on permuted labels; a leaky one scores well above 0.50.
# def leaky_score(y_use): ...


**Flaw table (your turn):** *(TODO — fill in ~5 rows.)*

| # | Flaw | Where in the analysis | Why it inflates / over-claims | Violates |
|---|------|------------------------|-------------------------------|----------|
| F1 | | | | |
| F2 | | | | |
| F3 | | | | |
| F4 | | | | |
| F5 | | | | |


---
## Part 3 — Propose corrections  *(≈20 min)*

For each flaw, state the principled fix and the *predicted* effect on the reported performance.


> **Exercise 3.1 — corrections table.** Complete the table: flaw → fix → predicted effect (most fixes should *lower* the headline number).

**Corrections table (your turn):** *(TODO)*

| Flaw | Fix | Predicted effect |
|------|-----|------------------|
| F1 | | |
| F2 | | |
| F3 | | |
| F4 | | |
| F5 | | |


---
## Part 4 — Build a corrected workflow  *(≈45 min — spine)*

Implement the fixes: selection and scaling **inside** the CV; **nested CV** for tuning; an honest
baseline on the engineered features; a **stability** check on interpretation; and a genuine **external
validation** on GSE6532 plus a **clinical-baseline** comparison.


> **Exercise 4.1 — leakage-safe internal estimate.** Build a `Pipeline` that does scaling → supervised
> selection → model, so *everything refits inside each fold*. Report its honest cross-validated AUC on the
> raw genes; compare to the headline.
>
> **Exercise 4.2 — nested CV for the tuned model.** Wrap the tuned gradient-boosting search in nested CV
> and report the honest estimate (removes the Part-1 tuning optimism).
>
> **Exercise 4.3 — external validation + clinical baseline.** Fit an honest baseline (regularised LR) on
> the **engineered features** (train), and evaluate on the **external** GSE6532 cohort. Compare to a
> clinical-style baseline (here: the `proliferation` signature alone, a grade/Ki-67 proxy).
>
> **Exercise 4.4 (optional) — stability & survival.** Check how often the top selected genes recur across
> resamples; and (optional) fit a conceptual survival model on the time-to-event outcome.


> **Real-world exemplar — ER-Predict.** The course's published exemplar, the **ER-Predict** assay
> (Boscolo Bielo et al., *ESMO Open* 2026), was developed on this same METABRIC cohort and made exactly
> the research-grade choice the 4.4 cell points at. Its core is **time-dependent (survival)** — a
> **survival SVM** plus a **gradient-boosting survival forest** — and a time-independent **binary
> classifier is used only as a tiebreaker** when the two survival models disagree. So the binary recurrence
> label you have used all course is, in the real model, demoted to a supporting role behind a survival core.
>
> **Scope note (unchanged):** we do not teach survival mechanics (hazard functions, partial likelihood).
> The goal is to *recognise* when a binary endpoint throws away information and to *know* that time-to-event
> methods exist and were the right choice for the real model.


In [ ]:
# TODO 4.1 / 4.2 / 4.3 / 4.4
# 4.1 Build a Pipeline(impute -> scale -> SelectKBest(k=50) -> LogisticRegression) so selection/scaling
#     refit INSIDE each fold. Report cross_val_score AUC vs the headline.
# 4.2 Wrap a tuned GradientBoosting pipeline in NESTED CV: cross_val_score(GridSearchCV(pipe, grid, cv=inner), ...).
# 4.3 Validate BOTH models on the REAL external cohort (GSE6532):
#     - the authors' FLAWED raw-gene model (use X_ext_genes[top50], scaled by the TRAIN scaler) -> watch it
#       collapse, because platform-specific probes do not transfer across cohorts;
#     - your CORRECTED engineered-feature model (batch-correct X_ext_feat onto the TRAIN reference) -> it
#       should HOLD, because biological signatures are cross-platform robust (Lecture 3).
#     Also compare to a clinical baseline (proliferation alone): the full model must BEAT it to add value.
# 4.4 Check selection stability across ~20 bootstraps (how many of 50 genes recur in >=80%?).
#     (Optional) fit a lifelines CoxPHFitter on surv_all to show the survival framing.
# CV = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


---
## Part 5 — Compare original vs corrected  *(≈25 min)*

Put the inflated original number next to the honest corrected numbers, and quantify how much was real
signal versus methodological artefact.


> **Exercise 5.1 — comparison.** Build a small table/figure: original (leaky) AUC vs corrected internal
> (leakage-safe + nested) vs corrected **external** AUC. Quantify the inflation (original − honest).
>
> **Exercise 5.2 — interpretation.** In one paragraph: how much of the original “performance” was real
> signal, and how much was leakage + tuning optimism + the absence of external validation?


In [ ]:
# TODO 5.1 / 5.2
# 5.1 Assemble a table/figure: original headline AUC; the FLAWED model's EXTERNAL AUC (raw genes, GSE6532);
#     corrected internal (leakage-safe); corrected internal (nested CV); corrected EXTERNAL (engineered,
#     GSE6532); and the clinical baseline.
# 5.2 Comment: the decisive test is EXTERNAL. How much of the original headline was leakage + tuning
#     optimism? Does the raw-gene signature transfer? Does the engineered model hold -- and does it beat
#     the clinical baseline (incremental value)?
# rows = [("original (leaky)", headline_auc), ("corrected internal", safe_auc), ...]


---
## Part 6 — Write a reviewer report  *(≈30 min — the primary deliverable)*

Using the reviewer checklist as a template, write a structured review of the **original** submission.
Sections: summary of the claim · major issues (with evidence) · minor issues · reproducibility assessment
· clinical-relevance assessment · **recommendation** (accept / minor revision / major revision / reject)
with justification. End with: *Would you recommend publication? Why or why not?*


**Reviewer report (your turn):**

*(TODO — write your structured review here, citing the evidence you gathered in Parts 2–5. Be specific:
point to the leakage, the missing external validation, the tuning optimism, and the unsupported causal
claims, and quantify the inflation. End with a clear recommendation and a one-paragraph justification.)*


---
## Part 7 — Reflection  *(≈15 min)*

A short written reflection (commit it to the notebook): which course lessons (L1–L5) were most important in
catching these flaws, and which do you expect to use most in your own research?


**Reflection (your turn):** *(TODO — a few sentences. Which lessons mattered most here? What will you carry into your own work?)*

---
### Deliverables checklist
- [ ] Reproduced headline number + restated claim (Part 1)
- [ ] Flaw table (5 flaws) + permuted-label leakage demonstration (Part 2)
- [ ] Corrections table with predicted effects (Part 3)
- [ ] Corrected workflow: leakage-safe internal, nested CV, external (GSE6532), clinical baseline, stability (Part 4)
- [ ] Original-vs-corrected comparison + inflation quantified (Part 5)
- [ ] Structured reviewer report with a justified recommendation (Part 6)
- [ ] Reflection on the course's key lessons (Part 7)

> **The message, in one line:** *the hardest part of ML is not training a model; it is demonstrating that
> the model is trustworthy.* You built the corrected workflow — and the reviewer report is where you prove
> you can also detect the absence of trust. That is the whole course, exercised once.
